# Unfair Receiver Analysis

In [ ]:
import pathlibimport osimport matplotlib.pyplot as pltimport numpy as npimport pandas as pdimport seaborn as snsfrom scipy import statsimport statsmodels.api as smfrom statsmodels.stats.multitest import multipletestssns.set_theme(style="whitegrid")if not pathlib.Path("pyproject.toml").exists():    os.chdir(pathlib.Path("../.."))RESULTS_DIR = pathlib.Path("results")PERSONAS = ["normal", "strategic", "greedy", "benevolent"]SOURCES = ["human", "computer"]

In [ ]:
def load_unfair_receiver(model):    """Load unfair_receiver data for a given model."""    dfs = []    for persona in PERSONAS:        for source in SOURCES:            subdir = "unfair_receiver" if source == "human" else "unfair_receiver_computer"            path = RESULTS_DIR / subdir / persona / model / "unfair_receiver_1.csv"            if path.exists():                dfs.append(pd.read_csv(path))            else:                print(f"Not found: {path}")    df = pd.concat(dfs, ignore_index=True)    df["decision"] = df["decision"].str.strip().str.lower()    df["accepted"] = (df["decision"] == "accept").astype(int)    df["is_computer"] = (df["proposal_source"] == "computer").astype(int)    bins = [0, 20, 30, 40, 50, 60, 100]    labels = ["~19", "20-29", "30-39", "40-49", "50-59", "60+"]    df["age_group"] = pd.cut(df["age"], bins=bins, labels=labels, right=False)    return df

## Part 1: Descriptive Analysis

In [ ]:
MODEL = "gpt-5-mini"FIGURES_DIR = pathlib.Path(f"analysis/figures/unfair_receiver/{MODEL}")FIGURES_DIR.mkdir(parents=True, exist_ok=True)df = load_unfair_receiver(MODEL)print(f"Model: {MODEL}, Total rows: {len(df)}")df.head()

### Accept Rate by Persona x Proposal Source

In [ ]:
pivot = df.groupby(["persona", "proposal_source"])["accepted"].agg(["mean", "count"])pivot.columns = ["accept_rate", "n"]print(pivot)fig, ax = plt.subplots(figsize=(9, 5))rate = df.groupby(["persona", "proposal_source"])["accepted"].mean().unstack()rate = rate.reindex(PERSONAS)rate[SOURCES].plot(kind="bar", ax=ax, color=["#4C72B0", "#DD8452"])ax.set_title("Accept Rate by Persona × Proposal Source", fontsize=14)ax.set_xlabel("Persona")ax.set_ylabel("Accept Rate")ax.set_ylim(0, 1)ax.legend(title="Proposal Source")for container in ax.containers:    ax.bar_label(container, fmt="%.2f", fontsize=9)plt.xticks(rotation=0)fig.tight_layout()fig.savefig(FIGURES_DIR / "accept_rate_by_persona_source.png", dpi=150)plt.show()

### Accept Rate by Gender x Proposal Source

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))gr = df.groupby(["gender", "proposal_source"])["accepted"].mean().unstack()gr[SOURCES].plot(kind="bar", ax=ax, color=["#4C72B0", "#DD8452"])ax.set_title("Accept Rate by Gender × Proposal Source", fontsize=14)ax.set_xlabel("Gender")ax.set_ylabel("Accept Rate")ax.set_ylim(0, 1)ax.legend(title="Proposal Source")for container in ax.containers:    ax.bar_label(container, fmt="%.2f", fontsize=9)plt.xticks(rotation=0)fig.tight_layout()fig.savefig(FIGURES_DIR / "accept_rate_by_gender_source.png", dpi=150)plt.show()

### Accept Rate by Age Group x Proposal Source

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))ar = df.groupby(["age_group", "proposal_source"])["accepted"].mean().unstack()ar[SOURCES].plot(kind="bar", ax=ax, color=["#4C72B0", "#DD8452"])ax.set_title("Accept Rate by Age Group × Proposal Source", fontsize=14)ax.set_xlabel("Age Group")ax.set_ylabel("Accept Rate")ax.set_ylim(0, 1)ax.legend(title="Proposal Source")for container in ax.containers:    ax.bar_label(container, fmt="%.2f", fontsize=9)plt.xticks(rotation=0)fig.tight_layout()fig.savefig(FIGURES_DIR / "accept_rate_by_age_source.png", dpi=150)plt.show()

### Accept Rate by Region x Proposal Source

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))rr = df.groupby(["region", "proposal_source"])["accepted"].mean().unstack()rr[SOURCES].plot(kind="bar", ax=ax, color=["#4C72B0", "#DD8452"])ax.set_title("Accept Rate by Region × Proposal Source", fontsize=14)ax.set_xlabel("Region")ax.set_ylabel("Accept Rate")ax.set_ylim(0, 1)ax.legend(title="Proposal Source")for container in ax.containers:    ax.bar_label(container, fmt="%.2f", fontsize=8)plt.xticks(rotation=45, ha="right")fig.tight_layout()fig.savefig(FIGURES_DIR / "accept_rate_by_region_source.png", dpi=150)plt.show()

### Overall & Detailed Summary

In [ ]:
print("=== Overall Accept Rate ===")print(df.groupby("proposal_source")["accepted"].agg(["mean", "count"]))print()print("=== Persona × Gender × Source ===")print(df.groupby(["persona", "gender", "proposal_source"])["accepted"].agg(["mean", "count"]))

## Part 2: Statistical Tests

In [ ]:
def chi2_test(data, label=""):    """Run chi-squared test on 2x2 table: source × accept/reject."""    ct = pd.crosstab(data["proposal_source"], data["accepted"])    chi2, p, dof, expected = stats.chi2_contingency(ct)    n = len(data)    cramers_v = np.sqrt(chi2 / n)    ct_arr = ct.values    if ct_arr.shape == (2, 2) and np.all(ct_arr > 0):        odds_ratio = (ct_arr[1, 1] * ct_arr[0, 0]) / (ct_arr[1, 0] * ct_arr[0, 1])    else:        odds_ratio = np.nan    return {        "label": label, "n": n,        "accept_human": f"{data[data['proposal_source']=='human']['accepted'].mean():.1%}",        "accept_computer": f"{data[data['proposal_source']=='computer']['accepted'].mean():.1%}",        "chi2": round(chi2, 3), "dof": dof, "p_value": p,        "cramers_v": round(cramers_v, 4),        "odds_ratio": round(odds_ratio, 3) if not np.isnan(odds_ratio) else "N/A",    }def sig_stars(p):    if p < 0.001: return "***"    elif p < 0.01: return "**"    elif p < 0.05: return "*"    return "n.s."

### Chi-squared Tests

In [ ]:
results = []results.append(chi2_test(df, "Overall"))for persona in PERSONAS:    results.append(chi2_test(df[df["persona"] == persona], f"Persona: {persona}"))for gender in sorted(df["gender"].unique()):    results.append(chi2_test(df[df["gender"] == gender], f"Gender: {gender}"))AGE_LABELS = ["~19", "20-29", "30-39", "40-49", "50-59", "60+"]for ag in AGE_LABELS:    sub = df[df["age_group"] == ag]    if len(sub) > 0:        results.append(chi2_test(sub, f"Age: {ag}"))for region in sorted(df["region"].unique()):    results.append(chi2_test(df[df["region"] == region], f"Region: {region}"))res_df = pd.DataFrame(results)reject_bonf, pvals_corrected, _, _ = multipletests(res_df["p_value"], method="bonferroni")res_df["p_bonferroni"] = pvals_correctedres_df["sig"] = res_df["p_value"].apply(sig_stars)res_df["sig_bonferroni"] = np.where(reject_bonf, "***", "n.s.")res_df

### Logistic Regression

In [ ]:
df_model = df.copy()dummies_persona = pd.get_dummies(df_model["persona"], prefix="persona", drop_first=True, dtype=float)dummies_gender = pd.get_dummies(df_model["gender"], prefix="gender", drop_first=True, dtype=float)dummies_age = pd.get_dummies(df_model["age_group"], prefix="age", drop_first=True, dtype=float)dummies_region = pd.get_dummies(df_model["region"], prefix="region", drop_first=True, dtype=float)# Model 1: Main effectsX1 = pd.concat([df_model[["is_computer"]], dummies_persona, dummies_gender, dummies_age, dummies_region], axis=1)X1 = sm.add_constant(X1)y = df_model["accepted"]model1 = sm.Logit(y, X1).fit(disp=False)print("=== Model 1: Main Effects ===")print(model1.summary2())or_df1 = pd.DataFrame({"OR": np.exp(model1.params), "CI_lower": np.exp(model1.conf_int()[0]),    "CI_upper": np.exp(model1.conf_int()[1]), "p_value": model1.pvalues, "sig": model1.pvalues.apply(sig_stars)})print("\nOdds Ratios:")or_df1.round(4)

### Model 2: Persona x Source Interaction

In [ ]:
for col in dummies_persona.columns:    df_model[f"interact_{col}"] = dummies_persona[col].values * df_model["is_computer"].valuesinteract_cols = [c for c in df_model.columns if c.startswith("interact_")]X2 = pd.concat([df_model[["is_computer"]], dummies_persona, dummies_gender, dummies_age, dummies_region, df_model[interact_cols]], axis=1)X2 = sm.add_constant(X2)model2 = sm.Logit(y, X2).fit(disp=False)print(model2.summary2())# LR testlr_stat = 2 * (model2.llf - model1.llf)lr_df = model2.df_model - model1.df_modellr_p = stats.chi2.sf(lr_stat, lr_df)print(f"\nLR test: stat={lr_stat:.4f}, df={lr_df}, p={lr_p:.4e} {sig_stars(lr_p)}")

### Model 3: Gender x Source Interaction

In [ ]:
df_model2 = df.copy()dummies_persona2 = pd.get_dummies(df_model2["persona"], prefix="persona", drop_first=True, dtype=float)dummies_gender2 = pd.get_dummies(df_model2["gender"], prefix="gender", drop_first=True, dtype=float)dummies_age2 = pd.get_dummies(df_model2["age_group"], prefix="age", drop_first=True, dtype=float)dummies_region2 = pd.get_dummies(df_model2["region"], prefix="region", drop_first=True, dtype=float)for col in dummies_gender2.columns:    df_model2[f"interact_{col}"] = dummies_gender2[col].values * df_model2["is_computer"].valuesinteract_cols2 = [c for c in df_model2.columns if c.startswith("interact_")]X3 = pd.concat([df_model2[["is_computer"]], dummies_persona2, dummies_gender2, dummies_age2, dummies_region2, df_model2[interact_cols2]], axis=1)X3 = sm.add_constant(X3)model3 = sm.Logit(y, X3).fit(disp=False)print(model3.summary2())lr_stat3 = 2 * (model3.llf - model1.llf)lr_df3 = model3.df_model - model1.df_modellr_p3 = stats.chi2.sf(lr_stat3, lr_df3)print(f"\nLR test (Model 1 vs 3): stat={lr_stat3:.4f}, df={lr_df3}, p={lr_p3:.4e} {sig_stars(lr_p3)}")